# Mobile Data Usage Prediction

**Dataset:** [Mobile Device Usage and User Behavior Dataset](https://www.kaggle.com/datasets/valakhorasani/mobile-device-usage-and-user-behavior-dataset)
**Target:** `Data Usage (MB/day)` — internet consumption per user per month

Run the cells top to bottom. The upload cell will ask you to pick the CSV
you downloaded from Kaggle.

## Step 0: Install packages & upload the dataset

In [ ]:
!pip install -q imbalanced-learn

In [ ]:
from google.colab import files
uploaded = files.upload()   # choose your Kaggle CSV, e.g. user_behavior_dataset.csv
input_filename = list(uploaded.keys())[0]
print("Loaded file:", input_filename)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)

df = pd.read_csv(input_filename)
TARGET = "Data Usage (MB/day)"   # change this if your column is named differently

print("Shape:", df.shape)
df.head()

## Step 0b: Drop identifier columns & classify column types

`User ID` is unique per row and carries no predictive signal, so it is dropped.
The remaining columns are split into **numeric** and **categorical** feature lists.

In [ ]:
id_cols = [c for c in df.columns if c.lower() in ("user id", "userid", "id")]
if id_cols:
    print("Dropping identifier column(s):", id_cols)
    df = df.drop(columns=id_cols)

categorical_cols = [c for c in df.select_dtypes(include=["object", "category"]).columns if c != TARGET]
numeric_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != TARGET]

print("Numeric feature columns    :", numeric_cols)
print("Categorical feature columns:", categorical_cols)
print("Target column              :", TARGET)

## Step 1: Handling missing values

- **Numeric columns** (App Usage Time, Screen On Time, Battery Drain, Number of Apps
  Installed, Age, and the target Data Usage) → filled with the **median**. The median is
  used instead of the mean because usage metrics are typically right-skewed (a few heavy
  users would otherwise pull the mean up).
- **Categorical columns** (Device Model, Operating System, Gender) → filled with the
  **mode** (most frequent category), since there's no numeric average for a category.

In [ ]:
print("Missing values before cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0])

for col in numeric_cols + [TARGET]:
    if df[col].isnull().any():
        med = df[col].median()
        n = df[col].isnull().sum()
        df[col] = df[col].fillna(med)
        print(f"  '{col}': filled {n} missing value(s) with median = {med:.2f}")

for col in categorical_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode()[0]
        n = df[col].isnull().sum()
        df[col] = df[col].fillna(mode_val)
        print(f"  '{col}': filled {n} missing value(s) with mode = '{mode_val}'")

print("\nMissing values remaining:", df.isnull().sum().sum())

## Step 2: Handling outliers (IQR capping)

Applied to every numeric column, **including the target**, before scaling — extreme
values would otherwise distort the mean/std that scaling relies on.

For each column: `Q1` = 25th percentile, `Q3` = 75th percentile, `IQR = Q3 - Q1`.
Values outside `[Q1 - 1.5·IQR, Q3 + 1.5·IQR]` are **capped** (not deleted), so every
user's record is kept — important with a dataset of only ~700 rows.

In [ ]:
outlier_cols = numeric_cols + [TARGET]
for col in outlier_cols:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = df[col].clip(lower=lower, upper=upper)
    print(f"  '{col}': {n_out} outlier(s) capped to [{lower:.2f}, {upper:.2f}]")

## Step 3: Encoding categorical variables

`Device Model`, `Operating System`, `Gender` are all **nominal** (no inherent order —
e.g. "iOS" isn't greater or less than "Android"), so they're **one-hot encoded**
(`drop_first=True` avoids the dummy-variable trap / perfect multicollinearity).
Label/ordinal encoding is deliberately avoided here, since turning categories into
0, 1, 2... would falsely imply an order to distance-based or linear models.

In [ ]:
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True) if categorical_cols else df.copy()
new_cols = [c for c in df_encoded.columns if c not in df.columns]
print("New one-hot columns created:", new_cols)
df_encoded.head()

## Step 4: Feature scaling

All numeric feature columns (App Usage Time, Screen On Time, Battery Drain, Number of
Apps Installed, Age, User Behavior Class) are scaled with **StandardScaler**
(`(x - mean) / std`), giving each a mean of 0 and std of 1. This matters because these
columns have very different natural ranges (Battery Drain in the thousands of mAh vs.
Age in tens of years) — without scaling, distance-based/gradient-based models (KNN, SVR,
regularized linear regression) would let large-magnitude columns dominate purely due to
units. The one-hot columns (already 0/1) and the target (kept in MB/day for
interpretable predictions) are **not** scaled.

In [ ]:
scaler = StandardScaler()
df_encoded[numeric_cols] = scaler.fit_transform(df_encoded[numeric_cols])
print("Mean of scaled columns (should be ~0):")
print(df_encoded[numeric_cols].mean().round(3))

## Step 5: Handling an imbalanced dataset

The target is **continuous**, so there are no classes in the classification sense —
but the target's *distribution* can still be imbalanced (e.g. far more light/medium
data users than extreme heavy users), which biases a model toward the common range.

1. Bin the target into 4 **equal-width** ranges (Low/Medium/High/Very High) to diagnose
   the real skew (equal-width, not equal-frequency, so genuine imbalance shows up).
2. **Stratify** the train/test split on those bins, so both sets keep the same mix of
   light vs. heavy users.
3. Apply **RandomOverSampler** to the training set only — rows from underrepresented
   bins are duplicated until every bin has a similar count. The test set is left
   untouched for honest evaluation.

In [ ]:
from imblearn.over_sampling import RandomOverSampler

y = df_encoded[TARGET]
X = df_encoded.drop(columns=[TARGET])

bins = pd.cut(y, bins=4, labels=["Low", "Medium", "High", "Very High"])
print("Target distribution (equal-width bins):")
print(bins.value_counts())

X_train, X_test, y_train, y_test, bins_train, bins_test = train_test_split(
    X, y, bins, test_size=0.2, random_state=42, stratify=bins
)
print("\nTrain-set bin distribution before balancing:")
print(bins_train.value_counts())

ros = RandomOverSampler(random_state=42)
X_train_bal, bins_train_bal = ros.fit_resample(X_train, bins_train)
y_train_bal = y_train.reset_index(drop=True).iloc[ros.sample_indices_].reset_index(drop=True)

print("\nTrain-set bin distribution after balancing:")
print(bins_train_bal.value_counts())

## Step 6: Save the final preprocessed files

Downloads two CSVs: the balanced training set and the untouched hold-out test set.

In [ ]:
final_train = X_train_bal.copy()
final_train[TARGET] = y_train_bal.values
final_train.to_csv("final_preprocessed_train.csv", index=False)

final_test = X_test.copy()
final_test[TARGET] = y_test.values
final_test.to_csv("final_preprocessed_test.csv", index=False)

print("Saved: final_preprocessed_train.csv", final_train.shape)
print("Saved: final_preprocessed_test.csv", final_test.shape)

from google.colab import files
files.download("final_preprocessed_train.csv")
files.download("final_preprocessed_test.csv")